# 01 — SAM3 → YOLO auto-labels (Fase 1)

**Destilación**: SAM3 (maestro) auto-etiqueta videos → dataset YOLO (alumno rápido/deployable).

- Frames extraídos equiespaciados por video. SAM3 text-prompt → logits 288 → **upscale bilinear** → máscara → **caja YOLO** normalizada.
- Clases detección: `robot`, `orange_ball`, `yellow_zone` (sin `green_floor`: caja de todo el frame, inútil).
- Split **por video** (anti-leak). Salida YOLO estándar + `data.yaml`.
- Videos: split NO-testing (fine-tuning + reserva). **NO** usar los 20 de testing (son eval/LoRA).

Toda la lógica vive en `sam3_yolo.py` (mismo folder).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))  # para importar sam3_yolo.py
import sam3_yolo as sy

REPO = Path('/workspace/FutBotMX-UAQTeam')
SAM3_PATH = REPO / 'assets' / 'sam3'
METADATA  = REPO / 'assets' / 'db_metadata.csv'
OUT_DIR   = Path.cwd() / 'yolo_dataset'   # notebooks/fase_1/yolo_dataset

# --- config ---
SPLITS_TRAIN = [1]        # 1=fine-tuning (23 vids). Agrega 0 (reserva, 80) para escalar.
FRAMES_PER_VIDEO = 30
VAL_FRAC = 0.2            # 20% de los VIDEOS a val (split por video)
SMOKE = 2                 # >0 = solo N videos para probar; None/0 = todos
CLASSES = sy.DEFAULT_CLASSES
print('clases:', [(c['id'], c['name'], c['prompt']) for c in CLASSES])

In [ ]:
model, processor, device = sy.load_sam3(SAM3_PATH)

In [ ]:
videos = sy.videos_by_split(METADATA, SPLITS_TRAIN)
print(f'videos en split {SPLITS_TRAIN}: {len(videos)}')
if SMOKE:
    videos = videos[:SMOKE]
    print(f'SMOKE -> usando {len(videos)} videos')

In [ ]:
res = sy.autolabel(
    videos, repo_root=REPO, out_dir=OUT_DIR,
    sam3_model=model, sam3_proc=processor, device=device,
    classes=CLASSES, frames_per_video=FRAMES_PER_VIDEO, val_frac=VAL_FRAC,
)
res

In [ ]:
# Sanity: dibuja las cajas YOLO sobre 4 imagenes al azar
import random, cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

names = [c['name'] for c in sorted(CLASSES, key=lambda c: c['id'])]
colors = [(255,80,80),(255,170,0),(255,235,0)]
img_dir = OUT_DIR / 'images' / 'train'
lbl_dir = OUT_DIR / 'labels' / 'train'
imgs = list(img_dir.glob('*.png'))
random.seed(0)
pick = random.sample(imgs, min(4, len(imgs)))
fig, axes = plt.subplots(1, len(pick), figsize=(5*len(pick), 5))
if len(pick)==1: axes=[axes]
for ax, ip in zip(axes, pick):
    im = np.array(Image.open(ip)); H,W = im.shape[:2]
    txt = (lbl_dir / (ip.stem+'.txt')).read_text().strip()
    for line in txt.splitlines():
        if not line: continue
        c,xc,yc,bw,bh = line.split(); c=int(c)
        xc,yc,bw,bh = float(xc)*W,float(yc)*H,float(bw)*W,float(bh)*H
        x0,y0 = int(xc-bw/2),int(yc-bh/2); x1,y1=int(xc+bw/2),int(yc+bh/2)
        cv2.rectangle(im,(x0,y0),(x1,y1),colors[c%3],3)
        cv2.putText(im,names[c],(x0,max(0,y0-5)),cv2.FONT_HERSHEY_SIMPLEX,0.8,colors[c%3],2)
    ax.imshow(im); ax.set_title(ip.name, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## Siguiente paso — entrenar YOLO (Luis Felipe)

Con el `data.yaml` generado:
```bash
pip install ultralytics
yolo detect train data=yolo_dataset/data.yaml model=yolo11n.pt epochs=100 imgsz=960
```
- Medir **mAP + FPS + VRAM** vs SAM3 (tabla de eficiencia para README).
- Quitar `SMOKE` y agregar `0` a `SPLITS_TRAIN` para escalar a los 103 videos no-testing.
- Eval honesto: correr sobre los **held-out** (`lora_split.csv`) que tienen GT manual.